In [220]:
path = '/cluster/project/sachan/ekaterina/uncertainty4reasoning/test_math_no_prm800k_Qwen3-8B_texts_uhead_prm_annotated'
N = 32

In [221]:
import os
os.chdir('/cluster/home/efadeeva/uncertainty4reasoning_gh')

In [222]:
from datasets import load_from_disk
dataset = load_from_disk(path)
assert len(dataset) % N == 0
dataset

Dataset({
    features: ['question', 'answer', 'input_ids', 'reply', 'MaximumSequenceProbability', 'MeanTokenEntropy', 'Perplexity', 'PTrue', 'JingweiNi/uhead_claim_Qwen3-8B_fixed_prm_layer1_dim512_head16_e5_lr5e-4_pos3', 'Qwen/Qwen2.5-Math-7B-PRM800K', 'Qwen/Qwen2.5-Math-PRM-7B', 'saved_stats', 'deepseek_anno'],
    num_rows: 160
})

In [223]:
rename_estimators = [
    ('MaximumSequenceProbability', 'Max Prob'),
    ('MeanTokenEntropy', 'Mean Token Entropy'),
    ('Perplexity', 'Perplexity'),
    ('PTrue', 'P(True)'),
    ('Qwen/Qwen2.5-Math-PRM-7B-PRM800K', 'Qwen2.5-Math-PRM-7B-PRM800K'),
    ('Qwen/Qwen2.5-Math-PRM-7B', 'Qwen2.5-Math-PRM-7B'),
    ('peiyi9979/math-shepherd-mistral-7b-prm', 'math-shepherd-mistral-7b-prm'),
    ('RLHFlow/Llama3.1-8B-PRM-Mistral-Data', 'Llama3.1-8B-PRM-Mistral-Data'),
    ('RLHFlow/Llama3.1-8B-PRM-Deepseek-Data', 'Llama3.1-8B-PRM-Deepseek-Data'),
    ('Skywork/Skywork-o1-Open-PRM-Qwen-2.5-1.5B', 'Skywork-o1-Open-PRM-Qwen-2.5-1.5B'),
    ('universalprm/Universal-PRM', 'Universal-PRM'),
    ('HuggingFaceH4/Qwen2.5-Math-1.5B-Instruct-PRM-0.2', 'Qwen2.5-Math-1.5B-Instruct-PRM-0.2'),
    ('JingweiNi/uhead_claim_Qwen3-8B_fixed_prm_layer1_dim512_head16_e5_lr5e-4_pos3', 'UHead deepseek-anno'),
    ('JingweiNi/uhead_claim_Qwen3-8B_self_fixed_prm_layer1_dim512_head16_e10_lr5e-4_pos3', 'UHead self-anno'),
]

In [ ]:
import numpy as np
import pandas as pd
from plot_utils import pretty_plot_table

data = {'method': [], 'accuracy': []}
all_accs = []
skipped = 0
for uq, rename_uq in rename_estimators:
    if uq not in dataset.column_names:
        continue
    accs = []
    for i in range(0, len(dataset), N):
        scores = dataset[uq][i:i + N]
        accuracies = [1 - x for x in dataset['deepseek_anno'][i:i + N]]
        if any([np.isnan(x) for x in accuracies]) or any([x is None or np.isnan(x) for x in scores]):
            skipped += 1
            continue
        all_accs += accuracies
        best_score = np.argmin(scores)
        accs.append(accuracies[best_score])
    data['method'].append(rename_uq)
    data['accuracy'].append(np.mean(accs).item())
print(f'Skipped {skipped} nans out of {len(dataset)} samples')
print('Mean accuracy:', np.mean(all_accs))
df = pd.DataFrame(data, index=None)
pretty_plot_table(df)